# Total tokens in dataset
Compute token counts for `train_pairs` and `eval_pairs` using `sentence-transformers/static-retrieval-mrl-en-v1`.

In [3]:
import os
import sys
from pathlib import Path

from transformers import AutoTokenizer
import json

path = Path("cost comparison data/")

input_10000_train = path / "input" / "train" / "pairs.train.jsonl"
input_10000_eval = path / "input" / "eval" / "pairs.eval.jsonl"

def load_jsonl(file_path: Path) -> list[dict]:
    with file_path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_pairs = load_jsonl(input_10000_train)
eval_pairs = load_jsonl(input_10000_eval)

all_pairs = train_pairs + eval_pairs

print(f"train_pairs: {len(train_pairs):,}")
print(f"eval_pairs: {len(eval_pairs):,}")
print(f"all_pairs: {len(all_pairs):,}")

/home/ab/Documents/repos/sagemaker-mlops-with-terraform/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train_pairs: 267,646
eval_pairs: 66,912
all_pairs: 334,558


In [4]:
MODEL_ID = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

def pair_token_total(pair: dict[str, str]) -> int:
    anchor_tokens = tokenizer(pair['anchor'], add_special_tokens=True, truncation=False)['input_ids']
    positive_tokens = tokenizer(pair['positive'], add_special_tokens=True, truncation=False)['input_ids']
    return len(anchor_tokens) + len(positive_tokens)

train_total_tokens = sum(pair_token_total(pair) for pair in train_pairs)
eval_total_tokens = sum(pair_token_total(pair) for pair in eval_pairs)
all_total_tokens = train_total_tokens + eval_total_tokens

train_avg_tokens = train_total_tokens / len(train_pairs) if train_pairs else 0
eval_avg_tokens = eval_total_tokens / len(eval_pairs) if eval_pairs else 0

print(f'Model: {MODEL_ID}')
print(f'Train total tokens: {train_total_tokens:,}')
print(f'Eval total tokens: {eval_total_tokens:,}')
print(f'All total tokens: {all_total_tokens:,}')
print(f'Train avg tokens per pair: {train_avg_tokens:.2f}')
print(f'Eval avg tokens per pair: {eval_avg_tokens:.2f}')

Token indices sequence length is longer than the specified maximum sequence length for this model (607 > 512). Running this sequence through the model will result in indexing errors


Model: sentence-transformers/all-MiniLM-L6-v2
Train total tokens: 51,095,856
Eval total tokens: 12,749,350
All total tokens: 63,845,206
Train avg tokens per pair: 190.91
Eval avg tokens per pair: 190.54


In [ ]:
output_file = path / "tokens_full.json"

with output_file.open("w", encoding="utf-8") as f:
    f.write(str(int(all_total_tokens)))

print(f"Saved token totals to: {output_file}")

Saved token totals to: cost comparison data/total_tokens.json
